# Experiment 3: SOTA DML approach 

Fully non-parametric model implemented by econml.dml.CasusalForestDML, with nuisance functions estimated using GradientBoostingRegressor and GradientBoostingClassifier from sklearn.ensemble.

In [1]:
# Import dependencies
import shap
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from prophet import Prophet
from datetime import timedelta
from scipy.stats import ks_2samp
from econml.dml import CausalForestDML
from sklearn.exceptions import DataConversionWarning
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
    
from source_code.utilities import *
from source_code.planograms_clustering_lib.planograms_clustering import PlanogramsClustering

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)  
warnings.filterwarnings('ignore', category=UserWarning)  

import logging
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

# Outcomes display settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set seed for reproducibility
SEED = 42    

/home/deepwsmyav/.pyenv/versions/python-3-9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


## Planograms clustering

In [2]:
times = []
times.append(time.perf_counter())    

# Fetch planograms that were active across the period January 1, 2022 to June 30, 2025
# Planograms, among other things, define the assortment of the product category under consideration
planograms = pd.read_csv('raw_data/planograms.csv', sep=';', 
                         dtype={'planogram_id': 'string', 'sku':'string', 'h_facings':'int', 'v_facings':'int', 'package_width':'float', 'package_height':'float', 'x_start':'float', 'y_start':'float'}, 
                         parse_dates=['from_date', 'to_date'])

# Perform planograms clustering 

# Prepare data according to the implementation in source_code/planograms_clustering_lib/planograms_clustering.py
X = {planogram_id: products_allocation[['sku', 'h_facings', 'v_facings', 'package_width', 'package_height', 'x_start', 'y_start']] \
    .rename(columns={'sku':'art_code', 'package_width':'unit_width', 'package_height':'unit_height'}) \
    .reset_index(drop=True)   
    for planogram_id, products_allocation in planograms.groupby('planogram_id')} 

# Build and fit the clustering model
pc = PlanogramsClustering()                 
pc.fit(X=X)                                 

display(pc.distance(shape_segment=0))          # Computed planogram distance matrix
# display(pc.labels.style.hide(axis='index'))    # Results of the clustering method

times.append(time.perf_counter())    

,510,835
510,0.000000,0.057522
835,0.057522,0.000000


## Data retrieving and processing

In [3]:
times.append(time.perf_counter())    

# Data available from January 1, 2022 to June 30, 2025
# The data were extracted based on the assortment specified by the planograms
sales = pd.read_csv('raw_data/sales.csv', sep=';', dtype={'sku': 'string', 'quantity_sold': 'float'}, parse_dates=['date_stamp'])
price_list = pd.read_csv('raw_data/price_list.csv', sep=';', dtype={'sku': 'string', 'selling_price': 'float'}, parse_dates=['date_stamp'])
promotions = pd.read_csv('raw_data/promotions.csv', sep=';', dtype={'sku': 'string', 'on_promotion': 'int'}, parse_dates=['date_stamp'])
opening_stocks = pd.read_csv('raw_data/opening_stocks.csv', sep=';', dtype={'sku': 'string', 'opening_stock': 'float'}, parse_dates=['date_stamp'])

# For each product ('sku'), the assortment periods defined in the planograms are expanded into daily time series to ensure consistency with the other data sources
product_assortment = (
    planograms[['sku', 'from_date', 'to_date']]                                                               # Select relevant columns
        .assign(
        to_date=lambda x: pd.to_datetime(x['to_date'].fillna('2025-06-30')),                                  # Fill missing dates 
        date_stamp=lambda x: [pd.date_range(s, e, freq='D') for s, e in zip(x['from_date'], x['to_date'])]    # Build daily ranges
    ).explode('date_stamp')[['sku', 'date_stamp']].drop_duplicates()                                          # Expand each date range into separate rows and keep only unique sku-date_stamp pairs
)

# Data merging
processed_data = (product_assortment.merge(opening_stocks, on=['sku', 'date_stamp'], how='left').fillna(0)    # Set missing 'opening_stock' to 0 for assortment products from planograms
                                      .merge(promotions, on=['sku', 'date_stamp'])
                                      .merge(price_list, on=['sku', 'date_stamp'])
                                      .merge(sales, on=['sku', 'date_stamp'], how='left'))
processed_data = processed_data[['date_stamp', 'sku', 'opening_stock', 'selling_price', 'on_promotion', 'quantity_sold']]
# The resulting data have the following columns: ['date_stamp', 'sku', 'opening_stock', 'selling_price', 'on_promotion', 'quantity_sold']

# Identify the closing days, i.e. where all SKUs have NaN in the 'quantity_sold' column
closing_days = (
    processed_data
    .groupby('date_stamp')['quantity_sold']
    .apply(lambda x: x.isna().all())    # True if all quantities are NaN for that date
    .where(lambda x: x)                 # Keep only True values
    .dropna()                           # Remove the rest
    .index.tolist()
)
# The identified dates correspond to actual closing days, namely New Year's Day, Easter, and Christmas
# closing_days: ['2022-01-01', '2022-04-17', '2022-12-25', '2023-01-01', '2023-04-09', '2023-12-25', '2024-01-01', '2024-03-31', '2024-12-25', '2025-01-01', '2025-04-20']

# Remove the closing days from data
processed_data = processed_data.loc[~processed_data['date_stamp'].isin(closing_days)]

# Replace missing values in 'quantity_sold' with 0 when there is available opening stock
processed_data.loc[(processed_data['opening_stock'] > 0) & (processed_data['quantity_sold'].isna()), 'quantity_sold'] = 0
# The result of this operation leaves 'quantity_sold' as NaN when 'opening_stock' = 0, i.e. if there is no product in stock then sales cannot occur

# Cases where 'quantity_sold' is NaN are not considered in our analysis and are therefore removed (no missing values are present in the other features)
processed_data = processed_data.dropna(subset=['quantity_sold'])

# Correct inventory errors, i.e. fix rows where 'opening_stock' < 'quantity_sold' by enforcing 'opening_stock' = 'quantity_sold'
processed_data['opening_stock'] = processed_data['opening_stock'].clip(lower=processed_data['quantity_sold'])

processed_data.reset_index(drop=True, inplace=True)

display(display_processed_data(processed_data))

times.append(time.perf_counter())    

date_stamp,sku,opening_stock,selling_price,on_promotion,quantity_sold
2022-01-02,10100,319,1.59,0,67
2022-01-03,10100,252,1.59,0,23
2022-01-04,10100,229,1.59,0,22
…,…,…,…,…,…
2025-06-27,10217,6,0.69,0,2
2025-06-28,10217,5,0.69,0,5
2025-06-30,10217,2,0.69,0,2


## Proposed data processing methodology

In [4]:
##############################################################################################################
#                                      START Data Processing Methodology                                     #
##############################################################################################################

times.append(time.perf_counter())    

# Minimum number of samples required per group (treated/untreated)
MIN_SAMPLES_PER_GROUP = 100

# Dictionary mapping candidate slaves to their potential master sets
slave_to_masterSet_mapping = {}    # Dict structure: {'slave': ['master', ...], ...}

all_skus = planograms['sku'].unique().tolist()    # Set of all involved SKUs

for slave in all_skus: 


##############################################################################################################
# PHASE 1: SLAVE-TO-MASTER IDENTIFICATION
##############################################################################################################

    slave_data = (
        processed_data
            .loc[(processed_data['sku'] == slave) & (processed_data['on_promotion'] == 0)]    # Keep only rows for the given slave SKU that are not on promotion
            .copy()
    )
    
    if len(slave_data) < 2*MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
        continue
        
    for master in set(all_skus) - set([slave]):    # Look for masters among all SKUs except the current slave
        
        master_data = (
            processed_data
                .loc[(processed_data['sku'] == master)]    # Keep only rows for the given master SKU
                .copy()
        )
        
        if len(master_data) < 2*MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
            
        slave_master_data = slave_data.merge(master_data, on='date_stamp', how='inner', suffixes=('_slave', '_master'))    # Merge slave and master data on date_stamp
 
        if (len(slave_master_data['on_promotion_master'].unique()) < 2) or slave_master_data['on_promotion_master'].value_counts().min() < MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
        
        _, p_value = ks_2samp(data1=slave_master_data.loc[slave_master_data['on_promotion_master'] == 0]['quantity_sold_slave'].values,
                              data2=slave_master_data.loc[slave_master_data['on_promotion_master'] == 1]['quantity_sold_slave'].values,
                              method='exact',
                              nan_policy='raise')
        
        # Check for different data distribution assuming a significance value of 0.05
        if p_value < 0.05:
            if slave not in slave_to_masterSet_mapping:
                slave_to_masterSet_mapping[slave] = [master]
            else:
                slave_to_masterSet_mapping[slave].append(master)
                
times.append(time.perf_counter())    

if not slave_to_masterSet_mapping:
    raise SystemExit("The method terminates at this stage without producing any result") from None

In [5]:
times.append(time.perf_counter())    

# Define seasonal features
seasonal_features = ['trend', 'weekly_seasonality', 'yearly_seasonality']

# Initialize the engineered slave feature dataset
engineered_slave_data = pd.DataFrame(columns=processed_data.columns.to_list()[:2] + seasonal_features + processed_data.drop(columns='on_promotion').columns.to_list()[2:])

for slave in list(slave_to_masterSet_mapping.keys()):   # Iterate over the set of slave candidates
    
##############################################################################################################
# PHASE 2: SEASONAL FEATURE EXTRACTION
##############################################################################################################
    
    ##########################################################################################################
    # SLAVE DATA SELECTION
    ##########################################################################################################
    slave_data = processed_data.loc[(processed_data['sku'] == slave) & (processed_data['on_promotion'] == 0)].copy()   # Remove promotions from slave data
    
    ##########################################################################################################
    # ANOMALY DETECTION 
    ##########################################################################################################
    # Perform anomaly detection via the quantile method (if CV > 0.33)    
    if len(slave_data['quantity_sold'].unique()) > 1 and slave_data['quantity_sold'].std(ddof=0) / slave_data['quantity_sold'].mean() >= 0.33:    # Skip if the series is constant (no outliers to remove)
        _lower_bound, _upper_bound = slave_data['quantity_sold'].quantile([0.005, 0.995]).tolist()           # Compute lower/upper thresholds at [0.005, 0.995] quantile range
        slave_data = slave_data[
            (slave_data['quantity_sold'] >= _lower_bound) & (slave_data['quantity_sold'] <= _upper_bound)    # Keep only values in the range [_lower_bound, _upper_bound]
        ].reset_index(drop=True)              
    
        if len(slave_data) < 2*MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
            
    ##########################################################################################################
    # TIME-SERIES DECOMPOSITION 
    ##########################################################################################################
    
    _days_in_month = 30.4375
    _first, _last = slave_data['date_stamp'].min(), slave_data['date_stamp'].max()
    _min_dt = slave_data['date_stamp'].diff().iloc[slave_data['date_stamp'].diff().values.nonzero()[0]].min()
    
    # Define the missing data constraint, i.e. skip if more than 50% of the time span is missing
    if not len(slave_data)/(_last - _first).days > 0.5:    
        # Update slave_to_masterSet_mapping by removing items that do not meet the missing data constraint
        slave_to_masterSet_mapping.pop(slave)
        continue
    
    # Define seasonality constraints, i.e. skip if time span is less than 2 years or if data has a minimum gap of 7 days or more
    if (_last - _first < pd.Timedelta(days=_days_in_month * 12 * 2)) or _min_dt >= pd.Timedelta(days=7):        
        # Update slave_to_masterSet_mapping by removing items for which seasonal decomposition could not be performed
        slave_to_masterSet_mapping.pop(slave)
        continue
        
    
    # Prepare data for Prophet
    _slave_data = slave_data.copy()
    
    # Define floor/cap constraints as required by Prophet framework
    _slave_data.insert(loc=len(_slave_data.columns)-1, column='floor', value=0)                                   # Negative sales are not possible
    _slave_data.insert(loc=len(_slave_data.columns)-1, column='cap', value=slave_data['opening_stock'].values)    # Sales cannot exceed the quantities defined by the opening stock
    _slave_data['cap'] = _slave_data['cap'].apply(lambda x: x if x > 0 else 1)
    
    _slave_data = _slave_data[['date_stamp', 'floor', 'cap', 'quantity_sold']].rename(columns={'date_stamp': 'ds', 'quantity_sold': 'y'})
    
    # Fit Prophet model    
    decomposition_model = Prophet(growth='logistic', 
                                  changepoint_range=0.90,
                                  yearly_seasonality=False,
                                  weekly_seasonality=False,
                                  daily_seasonality=False,
                                  uncertainty_samples=0)
    
    decomposition_model.add_seasonality(name=seasonal_features[1], period=7, fourier_order=3, mode='additive')                     # Add weekly seasonality
    decomposition_model.add_seasonality(name=seasonal_features[2], period=_days_in_month*12, fourier_order=10, mode='additive')    # Add yearly seasonality

    with ProphetUtility.suppress_stdout_stderr():
        decomposition_model.fit(df=_slave_data)
    
    # Compute seasonal decomposition for the current slave
    curr_seasonal_decomposition = decomposition_model.predict(_slave_data)[['ds'] + seasonal_features].rename(columns={'ds':'date_stamp'})
    
    ##########################################################################################################
    # EXTEND SLAVE DATA WITH SEASONAL FEATURES
    ##########################################################################################################
    # Build the engineered feature set for the current slave
    curr_engineered_slave_data = slave_data.merge(curr_seasonal_decomposition, on='date_stamp')[engineered_slave_data.columns]
     
    # Append the current slave features to the global engineered slave feature dataset
    engineered_slave_data = pd.concat([engineered_slave_data, curr_engineered_slave_data],ignore_index=True)

display(display_engineered_slave_data(engineered_slave_data))

times.append(time.perf_counter())   

if engineered_slave_data.empty:
    raise SystemExit("The method terminates at this stage without producing any result") from None

09:35:40 - cmdstanpy - INFO - Chain [1] start processing
09:35:40 - cmdstanpy - INFO - Chain [1] done processing
09:35:40 - cmdstanpy - INFO - Chain [1] start processing
09:35:40 - cmdstanpy - INFO - Chain [1] done processing
09:35:40 - cmdstanpy - INFO - Chain [1] start processing
09:35:40 - cmdstanpy - INFO - Chain [1] done processing
09:35:40 - cmdstanpy - INFO - Chain [1] start processing
09:35:40 - cmdstanpy - INFO - Chain [1] done processing
09:35:40 - cmdstanpy - INFO - Chain [1] start processing
09:35:40 - cmdstanpy - INFO - Chain [1] done processing
09:35:40 - cmdstanpy - INFO - Chain [1] start processing
09:35:40 - cmdstanpy - INFO - Chain [1] done processing
09:35:40 - cmdstanpy - INFO - Chain [1] start processing
09:35:41 - cmdstanpy - INFO - Chain [1] done processing
09:35:41 - cmdstanpy - INFO - Chain [1] start processing
09:35:41 - cmdstanpy - INFO - Chain [1] done processing
09:35:41 - cmdstanpy - INFO - Chain [1] start processing
09:35:41 - cmdstanpy - INFO - Chain [1]

date_stamp,sku,trend,weekly_seasonality,yearly_seasonality,opening_stock,selling_price,quantity_sold
2022-01-02,10100,11.38,1.87,20.19,319,1.59,67
2022-01-03,10100,8.98,-3.16,18.48,252,1.59,23
2022-01-04,10100,8.15,-5.18,16.73,229,1.59,22
…,…,…,…,…,…,…,…
2025-06-28,10219,0.06,0.09,0.24,4,1.45,0
2025-06-29,10219,0.43,0.10,0.18,28,1.45,0
2025-06-30,10219,0.43,0.08,0.13,28,1.45,0


In [6]:
times.append(time.perf_counter())    

slave_to_masterPromotionalPatterns = {}

for slave, masterSet in slave_to_masterSet_mapping.items():
    
##############################################################################################################
# PHASE 3: PROMOTIONAL PATTERNS IDENTIFICATION
##############################################################################################################
    
    relevant_dates = engineered_slave_data.loc[(engineered_slave_data['sku']==slave), 'date_stamp'].values    # Non-promotional dates for the slave
    _processed_data = processed_data.loc[processed_data['date_stamp'].isin(relevant_dates)]                   # Keep only rows with relevant dates
    
    master_promotional_patterns = (
        _processed_data
            .loc[lambda x: x['sku'].isin(masterSet) & (x['on_promotion'] == 1)]    # Keep only rows where SKU is in masterSet and on promotion
            .groupby('date_stamp', group_keys=False)['sku']                        # Group by date_stamp, collect corresponding SKUs
            .apply(frozenset)                                                      # Convert SKUs for each date into a frozenset
            .reset_index(drop=False)
    )
    
    master_promotional_patterns = (
        master_promotional_patterns
            .groupby('sku', as_index=False)['date_stamp']               # Group the dataframe by SKU
            .agg(len)                                                   # Count how many date_stamp entries each SKU has
            .loc[lambda x: x['date_stamp'] >= MIN_SAMPLES_PER_GROUP]    # Check min samples constraint
    )
    
    # Proceed only if valid promotional patterns have been identified
    if not master_promotional_patterns.empty:     
        
        _master_promotional_patterns = []
        for _pattern in master_promotional_patterns['sku'].tolist():    
            
            # Find all dates where all SKUs in _pattern are simultaneously not on promotion
            _nonPromotional_dates = (
                _processed_data
                    .loc[((_processed_data['sku'].isin(_pattern)) & (_processed_data['on_promotion'] == 0))]   
                    .groupby('date_stamp', group_keys=False)['sku']
                    .agg(len)
                    .loc[lambda x: x == len(_pattern)])
            
            if len(_nonPromotional_dates) >= MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
                _master_promotional_patterns.append(_pattern)
        
        if _master_promotional_patterns:
            slave_to_masterPromotionalPatterns[slave] = _master_promotional_patterns
        
times.append(time.perf_counter())  

if not slave_to_masterPromotionalPatterns:
    raise SystemExit("The method terminates at this stage without producing any result") from None

In [7]:
times.append(time.perf_counter())    

# Outcomes structure definition
detected_slave_masterPromotionalPattern_pairs = []

estimators = ['ATE', 'ATT', 'ATC']
metrics = ['estimate', 'ci_lower', 'ci_upper', 'std_err', 'z_stat', 'p_value']
all_cols = [('master_pattern',''), ('slave','')] + [(e, m) for e in estimators for m in metrics] + \
            [('ate_robustness',''), ('delta_sales_%',''), ('delta_revenue_€',''), ('trimmed_sample', ''), ('feature_importances','')]

causal_analysis = pd.DataFrame(columns=pd.MultiIndex.from_tuples(all_cols, names=['',''])).astype({
    ('master_pattern',''): 'object',
    ('slave',''): 'str',
    **{col: 'float' for col in [(e, m) for e in estimators for m in metrics]},
    ('ate_robustness',''): 'float',
    ('delta_sales_%',''): 'float',
    ('delta_revenue_€',''): 'float',
    ('trimmed_sample',''): 'bool',
    ('feature_importances',''): 'object'
})

# Perform causal analysis for each slave-masterPromotionalPattern pair
for slave, masterPromotionalPatterns in slave_to_masterPromotionalPatterns.items():
        
##############################################################################################################
# PHASE 4: EXPLANATORY VARIABLES DEFINITION
##############################################################################################################
    
    # Compute lagged 'quantity_sold' feature of slave data
    lagged_series = (engineered_slave_data
                         .loc[engineered_slave_data['sku'] == slave]     # Select data for the current slave 
                     [['date_stamp', 'quantity_sold']].copy()            # Keep 'date_stamp' and 'quantity_sold' columns
                    ) 
    lagged_series['date_stamp'] = lagged_series['date_stamp'] + timedelta(days=1)                    # Shift dates forward by 1 day
    lagged_series = lagged_series.rename(columns={'quantity_sold': f'quantity_sold_t-1_{slave}'})    # Rename 'quantity_sold' column in 'quantity_sold_t-1'
        
        
    for masterPromotionalPattern in masterPromotionalPatterns:
        
    ##########################################################################################################
    # EXTEND SLAVE DATA
    ##########################################################################################################
        # Select data for the current slave
        curr_engineered_slave_data = engineered_slave_data.loc[engineered_slave_data['sku'] == slave].copy()       
        # Rename features by appending the suffix _{slave}
        curr_engineered_slave_data = curr_engineered_slave_data.rename( 
            columns={col: f'{col}_{slave}' for col in curr_engineered_slave_data.columns if col not in ['date_stamp', 'sku']}
        )  
        
        # Add 'quantity_sold_t-1_{slave}' column to slave data
        curr_engineered_slave_data = curr_engineered_slave_data.merge(lagged_series, on='date_stamp', how='inner')               
        
    ##########################################################################################################
    # RETIEVE MASTER FEATURES
    ##########################################################################################################
        
        # Build 'quantity_sold_{master}' and 'on_promotion_{master}' columns
        masterPromotionalPattern = sorted(list(masterPromotionalPattern))  # Sort SKUs in alphanumeric order
        master_pattern_features = (
            processed_data.loc[processed_data['sku'].isin(masterPromotionalPattern)]                 # Select data for the current masterPromotionalPattern
              .pivot(index='date_stamp', columns='sku', values=['opening_stock', 'on_promotion'])    # Pivot to wide format with 'opening_stock' and 'on_promotion' columns
              .swaplevel(axis=1)                                
              .sort_index(axis=1, level=0)                      
        ) 
        master_pattern_features.columns = [f'{val}_{sku}' for sku, val in master_pattern_features.columns]    # Rename columns to 'quantity_sold_{sku}' and 'on_promotion_{sku}'
        master_pattern_features = master_pattern_features[sorted(master_pattern_features.columns)]            # For consistency, align columns according to the SKUs order in masterPromotionalPattern
        master_pattern_features = master_pattern_features.reset_index().rename(columns={'index': 'date_stamp'})

    ##########################################################################################################
    # DEFINE THE TREATMENT VARIABLE
    ##########################################################################################################
    
        # Build the 'treatment' column: set to 1 when all masters in the pattern are on promotion, 0 when none are on promotion. Drop mixed cases
        on_promotion_cols = [c for c in master_pattern_features.columns if c.startswith('on_promotion_')]   
        treated = (master_pattern_features[on_promotion_cols] == 1).all(axis=1)                              
        untreated = (master_pattern_features[on_promotion_cols] == 0).all(axis=1)
        master_pattern_features['treatment'] = np.where(treated, 1, np.where(untreated, 0, np.nan))    

    ##########################################################################################################
    # RESTRICT DATA TO RELEVANT DATES
    ##########################################################################################################
        
        master_pattern_features = master_pattern_features.dropna(subset=['treatment']).drop(columns=on_promotion_cols).reset_index(drop=True)
        master_pattern_features['treatment'] = master_pattern_features['treatment'].astype(int)
        # Enrich slave data with 'quantity_sold_{master}' and 'treatment' columns
        curr_engineered_slave_data = curr_engineered_slave_data.merge(master_pattern_features, on='date_stamp')
                                  

        if (len(curr_engineered_slave_data['treatment'].unique()) < 2) or curr_engineered_slave_data['treatment'].value_counts().min() < MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
        
    ##########################################################################################################
    # DEFINITION OF FEATURE SET X, TREATMENT VARIABLE T AND SALES Y
    ##########################################################################################################  
        # Identify the relevant features according to the EconML framework notation
        _X = curr_engineered_slave_data.drop(columns=['date_stamp', 'sku', 'treatment', f'quantity_sold_{slave}']).columns.tolist()
        _T = 'treatment'
        _Y = f'quantity_sold_{slave}'

        
##############################################################################################################
# PHASE 5: PROPENSITY-BASED SAMPLE TRIMMING
##############################################################################################################
        
        trimmed_sample = False
        skip_to_next = False
        result, scores = positivity_check(T=curr_engineered_slave_data[_T], 
                                          X=curr_engineered_slave_data[_X], 
                                          alpha=0.05,
                                          random_state=SEED)        
        
        while not result:
            
            n_samples_before = len(curr_engineered_slave_data)
            # Trimming operation
            curr_engineered_slave_data = curr_engineered_slave_data.loc[
                np.where(((scores >= 0.05)) & ((scores <= 0.95)))].reset_index(drop=True)
            
            # Stop if trimming no longer reduces the sample while positivity is still violated,
            # indicating that common support cannot be recovered for the current dataset
            if len(curr_engineered_slave_data) == n_samples_before:
                skip_to_next = True
                break
            
            trimmed_sample = True
            
            # Check min samples constraint
            if (len(curr_engineered_slave_data[_T].unique()) < 2
                or curr_engineered_slave_data[_T].value_counts().min() < MIN_SAMPLES_PER_GROUP):
                skip_to_next = True
                break
                
            result, scores = positivity_check(T=curr_engineered_slave_data[_T], 
                                              X=curr_engineered_slave_data[_X], 
                                              alpha=0.05,
                                              random_state=SEED)
        
        if skip_to_next:
            continue
            
##############################################################################################################
#                                       END Data Processing Methodology                                      #
##############################################################################################################
        
        detected_slave_masterPromotionalPattern_pairs.append((slave, masterPromotionalPattern))
    
##############################################################################################################
#                                               Causal Analysis                                              #
##############################################################################################################

        # Perform causal analysis using the Double Machine Learning (DML) procedure implemented by the EconML framework

        # Base heuristics for hyperparameters setting
        min_samples_leaf_gb = max(int(np.ceil(0.01 * len(curr_engineered_slave_data))), 2*len(_X))
        
        # Define model_y to fit the outcome (_Y) to the features (_X) in the DML procedure                 
        model_y = GradientBoostingRegressor(learning_rate=0.1,
                                            n_estimators=200,
                                            subsample=0.8,
                                            min_samples_split=int(2 * min_samples_leaf_gb),
                                            min_samples_leaf=min_samples_leaf_gb,
                                            max_depth=2,
                                            max_features=0.7, 
                                            random_state=SEED)  

        # Define model_t to fit the treatment (_T) to the features (_X) in the DML procedure
        model_t = GradientBoostingClassifier(learning_rate=0.1,
                                             n_estimators=200,
                                             subsample=0.8,
                                             min_samples_split=int(2 * min_samples_leaf_gb),
                                             min_samples_leaf=min_samples_leaf_gb,
                                             max_depth=2,
                                             max_features=0.7,
                                             random_state=SEED)
        
        # Base heuristics for hyperparameters setting
        min_samples_leaf_cf = max(30, int(np.sqrt(len(curr_engineered_slave_data))))

        # Define dml_estimator to fit the response residuals to the treatment residuals
        dml_estimator = CausalForestDML(model_y=model_y, 
                                        model_t=model_t, 
                                        categories=[0,1],
                                        cv=5,    # cross-fitting parameter
                                        discrete_treatment=True,
                                        n_estimators=200,
                                        max_depth=None, 
                                        min_samples_split=int(2 * min_samples_leaf_cf),
                                        min_samples_leaf=min_samples_leaf_cf,
                                        max_features='sqrt',
                                        random_state=SEED)
        
        # Train dml_estimator 
        dml_estimator.fit(Y=curr_engineered_slave_data[_Y],  
                          T=curr_engineered_slave_data[_T],  
                          X=curr_engineered_slave_data[_X], 
                          cache_values=True)
        
        # Discard the model if the orthogonality condition is violated, 
        # as required by the DML framework to ensure valid debiased causal estimation
        if not orthogonality_check(dml_estimator=dml_estimator, random_state=SEED):
            continue
        
        # Retrieve model diagnostics: ate_robustness value and feature importance mapping
        ate_robustness = np.round(dml_estimator.robustness_value(), 3)
        feature_importances = dict(zip(_X, np.round(dml_estimator.feature_importances_, 2))) 

        # Estimate the causal effects
        
        # Estimate ATE 
        _ate_inference = dml_estimator.ate_inference(X=curr_engineered_slave_data[_X])
        ate_estimate = _ate_inference.mean_point
        ate_ci_lower, ate_ci_upper = _ate_inference.conf_int_mean()
        ate_std_err, ate_z_stat, ate_p_value = _ate_inference.stderr_mean, _ate_inference.zstat(), _ate_inference.pvalue()

        # Estimate ATT
        _att_inference = dml_estimator.ate_inference(X=curr_engineered_slave_data.loc[curr_engineered_slave_data[_T]==1][_X])
        att_estimate = _att_inference.mean_point
        att_ci_lower, att_ci_upper = _att_inference.conf_int_mean()
        att_std_err, att_z_stat, att_p_value = _att_inference.stderr_mean, _att_inference.zstat(), _att_inference.pvalue()

        # Estimate ATC 
        _atc_inference = dml_estimator.ate_inference(X=curr_engineered_slave_data.loc[curr_engineered_slave_data[_T]==0][_X])
        atc_estimate = _atc_inference.mean_point
        atc_ci_lower, atc_ci_upper = _atc_inference.conf_int_mean()
        atc_std_err, atc_z_stat, atc_p_value = _atc_inference.stderr_mean, _atc_inference.zstat(), _atc_inference.pvalue()
        
        # Compute 'delta_sales_%' and 'delta_revenue_€' w.r.t. the inferred ate estimate
        mean_volume_t0 = curr_engineered_slave_data.loc[curr_engineered_slave_data[_T]==0, _Y].mean()    # Compute the mean sales of the slave product when untreated
        mean_price = curr_engineered_slave_data[f"selling_price_{slave}"].mean()
        
        delta_sales = np.round(ate_estimate/mean_volume_t0*100, 2)    # The following is the percent change computed as: {[(mean_volume_t0 + ate) - mean_volume_t0] / mean_volume_t0} * 100
        delta_revenue = np.round(mean_price*ate_estimate, 2)
        
        # Collect outcomes
        effects_and_diagnostics = [ate_estimate, ate_ci_lower, ate_ci_upper, ate_std_err, ate_z_stat, ate_p_value, 
                                   att_estimate, att_ci_lower, att_ci_upper, att_std_err, att_z_stat, att_p_value,
                                   atc_estimate, atc_ci_lower, atc_ci_upper, atc_std_err, atc_z_stat, atc_p_value]
        effects_and_diagnostics = [np.round(val, 3) if ii in [5, 11, 17] else np.round(val, 2) for ii, val in enumerate(effects_and_diagnostics)]    # Round p_values to 3 decimals, others to 2
        curr_causal_analysis = [[masterPromotionalPattern, slave] + effects_and_diagnostics + 
                                [ate_robustness, delta_sales, delta_revenue, 
                                 trimmed_sample, feature_importances]]
        
        # Append the current causal analysis to the global outcomes structure
        curr_causal_analysis = pd.DataFrame(curr_causal_analysis, columns=causal_analysis.columns)    
        causal_analysis = pd.concat([causal_analysis, curr_causal_analysis], ignore_index=True)    
        
        
##############################################################################################################
#                                              START Shap Analysis                                           #
##############################################################################################################
        
        # Shap analysis is performed only when at least one causal effect (ATE, ATT, ATC) 
        # is statistically significant
        significance_level = 0.05
        significant_ate = (ate_p_value < significance_level) & (np.sign(ate_ci_lower*ate_ci_upper) == 1)
        significant_att = (att_p_value < significance_level) & (np.sign(att_ci_lower*att_ci_upper) == 1)
        significant_atc = (atc_p_value < significance_level) & (np.sign(atc_ci_lower*atc_ci_upper) == 1)

        if (significant_ate | significant_att | significant_atc):
            
            def _cate_predict(X_input):
                return dml_estimator.effect(X_input)

            # Select a background dataset for SHAP
            X_background = shap.kmeans(curr_engineered_slave_data[_X], 20)
    
            # Define Explainer
            explainer = shap.KernelExplainer(model=_cate_predict, data=X_background)

            # Compute SHAP values on a subset of observations
            rng = np.random.RandomState(SEED)
            n_explain = min(200, len(curr_engineered_slave_data))
            idx_explain = rng.choice(len(curr_engineered_slave_data), size=n_explain, replace=False)
            X_explain = curr_engineered_slave_data.loc[idx_explain][_X].copy()

            shap_values = explainer.shap_values(X_explain)
            shap_explanation = shap.Explanation(values=shap_values, data=X_explain, feature_names=_X)

            plt.figure(figsize=(10, 6))
            shap.plots.beeswarm(shap_explanation, max_display=len(_X), show=False)
            plt.xlabel("SHAP value")
            plt.tight_layout()

            _rule_id = generate_rule_id("_".join(masterPromotionalPattern + [slave]))
            plt.savefig(f"outcomes/sotadml_shap_analysis/sotadml_shap_analysis_rule_id_{_rule_id}.pdf", dpi=300, bbox_inches="tight")
            plt.close() 
            
##############################################################################################################
#                                              END Shap Analysis                                             #
##############################################################################################################
 
    
times.append(time.perf_counter())    
print('Detected master pattern-slave pairs:', len(detected_slave_masterPromotionalPattern_pairs))
print(   'Experiment duration (in minutes):', f"{np.round(sum([times[i+1] - times[i] for i in range(0, len(times)-1, 2)])/60,2):.2f}")
    
if causal_analysis.empty:
    raise SystemExit("The method terminates without producing any result") from None

100%|██████████| 200/200 [00:39<00:00,  5.10it/s]


Detected master pattern-slave pairs: 52
Experiment duration (in minutes): 9.91


### Causal analysis outcomes

In [8]:
outcomes = None

if not causal_analysis.empty:
    # Filter the significant estimated causal effects, i.e. those with p_value < 0.05 and with confidence intervals that have the same sign, in at least one of the considered metrics (ATE, ATT, ATC)
    significance_level = 0.05
    significant_ate = (causal_analysis.ATE.p_value < significance_level) & (np.sign(causal_analysis.ATE.ci_lower*causal_analysis.ATE.ci_upper) == 1)
    significant_att = (causal_analysis.ATT.p_value < significance_level) & (np.sign(causal_analysis.ATT.ci_lower*causal_analysis.ATT.ci_upper) == 1)
    significant_atc = (causal_analysis.ATC.p_value < significance_level) & (np.sign(causal_analysis.ATC.ci_lower*causal_analysis.ATC.ci_upper) == 1)

    # Invalid not significant outcomes
    causal_analysis.loc[~significant_ate, pd.IndexSlice["ATE", :]] = np.nan
    causal_analysis.loc[~significant_ate, pd.IndexSlice["ate_robustness"]] = np.nan 
    causal_analysis.loc[~significant_ate, pd.IndexSlice["delta_sales_%"]] = np.nan 
    causal_analysis.loc[~significant_ate, pd.IndexSlice["delta_revenue_€"]] = np.nan 
    causal_analysis.loc[~significant_att, pd.IndexSlice["ATT", :]] = np.nan
    causal_analysis.loc[~significant_atc, pd.IndexSlice["ATC", :]] = np.nan
    causal_analysis.loc[(~significant_ate) & (~significant_att) & (~significant_atc), 
                        pd.IndexSlice["trimmed_sample"]] = np.nan
    causal_analysis.loc[(~significant_ate) & (~significant_att) & (~significant_atc), 
                        pd.IndexSlice["feature_importances"]] = np.nan

    outcomes = causal_analysis.loc[(significant_ate | significant_att | significant_atc)]
    outcomes = outcomes.reset_index(drop=True)

    # Sort by master_pattern, slave
    outcomes['_tmp_master_pattern'] = outcomes[('master_pattern', '')].apply(str)
    outcomes = outcomes.sort_values(by=['_tmp_master_pattern', ('slave', '')])   
    outcomes = outcomes.drop(columns='_tmp_master_pattern', level=0)

    # Assign rule_id
    outcomes[("rule_id", "")] = outcomes.apply(
        lambda row: generate_rule_id("_".join(map(str, row[("master_pattern", "")] + [row[("slave", "")]]))),
        axis=1
    )
    outcomes.set_index("rule_id", inplace=True)

    # Store outcomes
    outcomes.to_pickle("outcomes/sotadml_outcomes.pkl")

    # Retrieve sku descriptions from planograms
    descriptions = (
        planograms.loc[planograms['sku'].isin(set(outcomes['master_pattern'].explode()) | set(outcomes['slave'])),
                       ['sku', 'description']]
                    .drop_duplicates()
                    .sort_values('sku')
                    .reset_index(drop=True))

    print('         Total estimated effects:', f"{len(causal_analysis):>4d}")
    print('   Significant estimated effects:', f"{len(outcomes):>4d}")
    print("\nNOTE: The significant estimated causal effects are those with p_value < 0.05 and with confidence intervals that have the same sign, in at least one of the considered metrics (ATE, ATT, ATC).")

    # display(outcomes)
    # display(descriptions)

         Total estimated effects:   41
   Significant estimated effects:    6

NOTE: The significant estimated causal effects are those with p_value < 0.05 and with confidence intervals that have the same sign, in at least one of the considered metrics (ATE, ATT, ATC).


In [10]:
if outcomes is not None and not outcomes.empty:
    # Display results by replacing SKUs with their corresponding descriptions 
    results_with_desc = outcomes.copy()
    sku_to_desc = dict(zip(descriptions['sku'].astype(str), descriptions['description']))      # Build mapping {sku: description}
    results_with_desc['master_pattern'] = results_with_desc['master_pattern'].apply(           # Replace SKUs in master_pattern with descriptions
        lambda x: [sku_to_desc[master] for master in x])
    results_with_desc['slave'] = results_with_desc['slave'].apply(lambda x: sku_to_desc[x])    # Replace SKUs in slave with descriptions
    display(results_with_desc)

master_pattern  \
                                                                                                                                                                          
rule_id                                                                                                                                                                   
Q5G                                                                                                                    [COCA COLA PET 1,5LT, COCA COLA ZERO PET LT1,5£]   
D30                                                                                                                    [COCA COLA PET 1,5LT, COCA COLA ZERO PET LT1,5£]   
W6M                                                                                                                                                    [COCA COLA LT.2]   
BV9                                                                                      [PEPSI REGULAR 1,5L.PET, PEPSI MAX S/CAFFEINA PET LT.1,5, PEPSI TWIST LT.1,5£]   
Z4R      [BEN COLA S.BENEDETTO PET LT1,5£, BIBITA ALLEGRA S.BENED.LT1,5, LIMONATA S.BENEDETTO PT LT.1,5, GASSOSA S.BENEDETTO PET LT.1,5, CEDRATA S.BENEDETTO PET LT1,5]   
0SE                                                                                                                                             [FANTA ORANGE PET CL66]   

                               slave      ATE                            \
                                     estimate ci_lower ci_upper std_err   
rule_id                                                                   
Q5G      COCA COLA PET LT1,35X2 BPK£    -3.11    -4.65    -1.57    0.78   
D30               COCA COLA LT1 PET£    -2.03    -3.27    -0.79    0.63   
W6M           PEPSI REGULAR 1,5L.PET    -2.33    -4.11    -0.55    0.91   
BV9          PEPSI COLA REG.PET CL50    -1.04    -1.91    -0.17    0.44   
Z4R        PEPSI-COLA BARATTOLO CL33      NaN      NaN      NaN     NaN   
0SE                 SPRITE LT1,5 PET    -0.86    -1.71    -0.01    0.43   

                            ATT                                           \
        z_stat p_value estimate ci_lower ci_upper std_err z_stat p_value   
rule_id                                                                    
Q5G      -3.97   0.000    -2.96    -4.57    -1.35    0.82  -3.61   0.000   
D30      -3.21   0.001    -2.13    -3.31    -0.95    0.60  -3.53   0.000   
W6M      -2.57   0.010    -2.41    -3.96    -0.87    0.79  -3.06   0.002   
BV9      -2.35   0.019    -1.06    -1.78    -0.33    0.37  -2.85   0.004   
Z4R        NaN     NaN     1.73     0.15     3.32    0.81   2.15   0.032   
0SE      -1.98   0.047      NaN      NaN      NaN     NaN    NaN     NaN   

             ATC                                          ate_robustness  \
        estimate ci_lower ci_upper std_err z_stat p_value                  
rule_id                                                                    
Q5G        -3.15    -4.67    -1.64    0.77  -4.07   0.000          0.101   
D30        -2.01    -3.26    -0.75    0.64  -3.13   0.002          0.057   
W6M        -2.30    -4.17    -0.43    0.95  -2.41   0.016          0.048   
BV9        -1.04    -1.94    -0.14    0.46  -2.27   0.023          0.029   
Z4R          NaN      NaN      NaN     NaN    NaN     NaN            NaN   
0SE        -0.85    -1.69    -0.01    0.43  -1.99   0.047          0.024   

        delta_sales_% delta_revenue_€ trimmed_sample  \
                                                       
rule_id                                                
Q5G            -26.33           -9.53           True   
D30            -20.30           -3.30           True   
W6M            -17.30           -3.30           True   
BV9            -28.70           -0.88           True   
Z4R               NaN             NaN          False   
0SE            -49.49           -1.41          False   

                                                              

## Placebo test

The placebo test is conducted under the following hypotheses.

**Null hypothesis (H0):**
The observed causal effect is drawn from the null distribution of effects
obtained by permuting the treatment T; no real causal effect exists.

**Alternative hypothesis (H1):**
The observed causal effect does not belong to the null distribution and
indicates a real causal effect (positive or negative).

NOTE: The p-value is the two-sided probability, under the null distribution,
of observing an effect at least as extreme as obs_estimate.

In [11]:
times = []

placebo_outcomes = None

try:
    outcomes = pd.read_pickle('outcomes/sotadml_outcomes.pkl')
    
except:
    raise SystemExit("No valid causal estimates were produced by the DML procedure. The placebo test is therefore not executed.") from None

times.append(time.perf_counter())  

##############################################################################################################
#                                      START Data Processing Methodology                                     #
##############################################################################################################

# Minimum number of samples required per group (treated/untreated)
MIN_SAMPLES_PER_GROUP = 100

# Dictionary mapping candidate slaves to their potential master sets
slave_to_masterSet_mapping = {}    # Dict structure: {'slave': ['master', ...], ...}

all_skus = planograms['sku'].unique().tolist()    # Set of all involved SKUs

for slave in all_skus: 


##############################################################################################################
# PHASE 1: SLAVE-TO-MASTER IDENTIFICATION
##############################################################################################################

    slave_data = (
        processed_data
            .loc[(processed_data['sku'] == slave) & (processed_data['on_promotion'] == 0)]    # Keep only rows for the given slave SKU that are not on promotion
            .copy()
    )
    
    if len(slave_data) < 2*MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
        continue
        
    for master in set(all_skus) - set([slave]):    # Look for masters among all SKUs except the current slave
        
        master_data = (
            processed_data
                .loc[(processed_data['sku'] == master)]    # Keep only rows for the given master SKU
                .copy()
        )
        
        if len(master_data) < 2*MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
            
        slave_master_data = slave_data.merge(master_data, on='date_stamp', how='inner', suffixes=('_slave', '_master'))    # Merge slave and master data on date_stamp
 
        if (len(slave_master_data['on_promotion_master'].unique()) < 2) or slave_master_data['on_promotion_master'].value_counts().min() < MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
        
        _, p_value = ks_2samp(data1=slave_master_data.loc[slave_master_data['on_promotion_master'] == 0]['quantity_sold_slave'].values,
                              data2=slave_master_data.loc[slave_master_data['on_promotion_master'] == 1]['quantity_sold_slave'].values,
                              method='exact',
                              nan_policy='raise')
        
        # Check for different data distribution assuming a significance value of 0.05
        if p_value < 0.05:
            if slave not in slave_to_masterSet_mapping:
                slave_to_masterSet_mapping[slave] = [master]
            else:
                slave_to_masterSet_mapping[slave].append(master)
                
if not slave_to_masterSet_mapping:
    raise SystemExit("The test terminates at Phase 1 without producing any result") from None
    
# Define seasonal features
seasonal_features = ['trend', 'weekly_seasonality', 'yearly_seasonality']

# Initialize the engineered slave feature dataset
engineered_slave_data = pd.DataFrame(columns=processed_data.columns.to_list()[:2] + seasonal_features + processed_data.drop(columns='on_promotion').columns.to_list()[2:])

for slave in list(slave_to_masterSet_mapping.keys()):   # Iterate over the set of slave candidates
    
##############################################################################################################
# PHASE 2: SEASONAL FEATURE EXTRACTION
##############################################################################################################
    
    ##########################################################################################################
    # SLAVE DATA SELECTION
    ##########################################################################################################
    slave_data = processed_data.loc[(processed_data['sku'] == slave) & (processed_data['on_promotion'] == 0)].copy()   # Remove promotions from slave data
    
    ##########################################################################################################
    # ANOMALY DETECTION 
    ##########################################################################################################
    # Perform anomaly detection via the quantile method (if CV > 0.33)    
    if len(slave_data['quantity_sold'].unique()) > 1 and slave_data['quantity_sold'].std(ddof=0) / slave_data['quantity_sold'].mean() >= 0.33:    # Skip if the series is constant (no outliers to remove)
        _lower_bound, _upper_bound = slave_data['quantity_sold'].quantile([0.005, 0.995]).tolist()           # Compute lower/upper thresholds at [0.005, 0.995] quantile range
        slave_data = slave_data[
            (slave_data['quantity_sold'] >= _lower_bound) & (slave_data['quantity_sold'] <= _upper_bound)    # Keep only values in the range [_lower_bound, _upper_bound]
        ].reset_index(drop=True)              
    
        if len(slave_data) < 2*MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
            
    ##########################################################################################################
    # TIME-SERIES DECOMPOSITION 
    ##########################################################################################################
    
    _days_in_month = 30.4375
    _first, _last = slave_data['date_stamp'].min(), slave_data['date_stamp'].max()
    _min_dt = slave_data['date_stamp'].diff().iloc[slave_data['date_stamp'].diff().values.nonzero()[0]].min()
    
    # Define the missing data constraint, i.e. skip if more than 50% of the time span is missing
    if not len(slave_data)/(_last - _first).days > 0.5:    
        # Update slave_to_masterSet_mapping by removing items that do not meet the missing data constraint
        slave_to_masterSet_mapping.pop(slave)
        continue
    
    # Define seasonality constraints, i.e. skip if time span is less than 2 years or if data has a minimum gap of 7 days or more
    if (_last - _first < pd.Timedelta(days=_days_in_month * 12 * 2)) or _min_dt >= pd.Timedelta(days=7):        
        # Update slave_to_masterSet_mapping by removing items for which seasonal decomposition could not be performed
        slave_to_masterSet_mapping.pop(slave)
        continue
        
    
    # Prepare data for Prophet
    _slave_data = slave_data.copy()
    
    # Define floor/cap constraints as required by Prophet framework
    _slave_data.insert(loc=len(_slave_data.columns)-1, column='floor', value=0)                                   # Negative sales are not possible
    _slave_data.insert(loc=len(_slave_data.columns)-1, column='cap', value=slave_data['opening_stock'].values)    # Sales cannot exceed the quantities defined by the opening stock
    _slave_data['cap'] = _slave_data['cap'].apply(lambda x: x if x > 0 else 1)
    
    _slave_data = _slave_data[['date_stamp', 'floor', 'cap', 'quantity_sold']].rename(columns={'date_stamp': 'ds', 'quantity_sold': 'y'})
    
    # Fit Prophet model    
    decomposition_model = Prophet(growth='logistic', 
                                  changepoint_range=0.90,
                                  yearly_seasonality=False,
                                  weekly_seasonality=False,
                                  daily_seasonality=False,
                                  uncertainty_samples=0)
    
    decomposition_model.add_seasonality(name=seasonal_features[1], period=7, fourier_order=3, mode='additive')                     # Add weekly seasonality
    decomposition_model.add_seasonality(name=seasonal_features[2], period=_days_in_month*12, fourier_order=10, mode='additive')    # Add yearly seasonality

    with ProphetUtility.suppress_stdout_stderr():
        decomposition_model.fit(df=_slave_data)
    
    # Compute seasonal decomposition for the current slave
    curr_seasonal_decomposition = decomposition_model.predict(_slave_data)[['ds'] + seasonal_features].rename(columns={'ds':'date_stamp'})
    
    ##########################################################################################################
    # EXTEND SLAVE DATA WITH SEASONAL FEATURES
    ##########################################################################################################
    # Build the engineered feature set for the current slave
    curr_engineered_slave_data = slave_data.merge(curr_seasonal_decomposition, on='date_stamp')[engineered_slave_data.columns]
     
    # Append the current slave features to the global engineered slave feature dataset
    engineered_slave_data = pd.concat([engineered_slave_data, curr_engineered_slave_data],ignore_index=True)

if engineered_slave_data.empty:
    raise SystemExit("The test terminates at Phase 2 without producing any result") from None


slave_to_masterPromotionalPatterns = {}

for slave, masterSet in slave_to_masterSet_mapping.items():
    
##############################################################################################################
# PHASE 3: PROMOTIONAL PATTERNS IDENTIFICATION
##############################################################################################################
    
    relevant_dates = engineered_slave_data.loc[(engineered_slave_data['sku']==slave), 'date_stamp'].values    # Non-promotional dates for the slave
    _processed_data = processed_data.loc[processed_data['date_stamp'].isin(relevant_dates)]                   # Keep only rows with relevant dates
    
    master_promotional_patterns = (
        _processed_data
            .loc[lambda x: x['sku'].isin(masterSet) & (x['on_promotion'] == 1)]    # Keep only rows where SKU is in masterSet and on promotion
            .groupby('date_stamp', group_keys=False)['sku']                        # Group by date_stamp, collect corresponding SKUs
            .apply(frozenset)                                                      # Convert SKUs for each date into a frozenset
            .reset_index(drop=False)
    )
    
    master_promotional_patterns = (
        master_promotional_patterns
            .groupby('sku', as_index=False)['date_stamp']               # Group the dataframe by SKU
            .agg(len)                                                   # Count how many date_stamp entries each SKU has
            .loc[lambda x: x['date_stamp'] >= MIN_SAMPLES_PER_GROUP]    # Check min samples constraint
    )
    
    # Proceed only if valid promotional patterns have been identified
    if not master_promotional_patterns.empty:     
        
        _master_promotional_patterns = []
        for _pattern in master_promotional_patterns['sku'].tolist():    
            
            # Find all dates where all SKUs in _pattern are simultaneously not on promotion
            _nonPromotional_dates = (
                _processed_data
                    .loc[((_processed_data['sku'].isin(_pattern)) & (_processed_data['on_promotion'] == 0))]   
                    .groupby('date_stamp', group_keys=False)['sku']
                    .agg(len)
                    .loc[lambda x: x == len(_pattern)])
            
            if len(_nonPromotional_dates) >= MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
                _master_promotional_patterns.append(_pattern)
        
        if _master_promotional_patterns:
            slave_to_masterPromotionalPatterns[slave] = _master_promotional_patterns
        
if not slave_to_masterPromotionalPatterns:
    raise SystemExit("The test terminates at Phase 3 without producing any result") from None

# Placebo Outcomes structure definition
estimators = ['ATE', 'ATT', 'ATC']
metrics = [
    'estimate',
    'placebo_estimate',
    'placebo_ci_lower',
    'placebo_ci_upper',
    'placebo_p_value'
]

all_cols = (
    [('master_pattern',''), ('slave','')] +
    [(e, m) for e in estimators for m in metrics]
)

placebo_outcomes = pd.DataFrame(
    columns=pd.MultiIndex.from_tuples(all_cols)
).astype({
    ('master_pattern',''): 'object',
    ('slave',''): 'object',
    **{(e, m): 'float' for e in estimators for m in metrics}
})

# Override the slave–masterPromotionalPattern data structure with the subset of pairs
# for which a statistically significant causal effect was estimated in current experiment
slave_to_masterPromotionalPatterns = {
    s: [frozenset(mp) for mp in outcomes.loc[
        outcomes['slave'] == s, 'master_pattern'
    ]] for s in outcomes['slave'].unique()
}

# Perform causal analysis for each slave-masterPromotionalPattern pair
for slave, masterPromotionalPatterns in slave_to_masterPromotionalPatterns.items():
        
##############################################################################################################
# PHASE 4: EXPLANATORY VARIABLES DEFINITION
##############################################################################################################
    
    # Compute lagged 'quantity_sold' feature of slave data
    lagged_series = (engineered_slave_data
                         .loc[engineered_slave_data['sku'] == slave]     # Select data for the current slave 
                     [['date_stamp', 'quantity_sold']].copy()            # Keep 'date_stamp' and 'quantity_sold' columns
                    ) 
    lagged_series['date_stamp'] = lagged_series['date_stamp'] + timedelta(days=1)                    # Shift dates forward by 1 day
    lagged_series = lagged_series.rename(columns={'quantity_sold': f'quantity_sold_t-1_{slave}'})    # Rename 'quantity_sold' column in 'quantity_sold_t-1'
        
        
    for masterPromotionalPattern in masterPromotionalPatterns:
        
    ##########################################################################################################
    # EXTEND SLAVE DATA
    ##########################################################################################################
        # Select data for the current slave
        curr_engineered_slave_data = engineered_slave_data.loc[engineered_slave_data['sku'] == slave].copy()       
        # Rename features by appending the suffix _{slave}
        curr_engineered_slave_data = curr_engineered_slave_data.rename( 
            columns={col: f'{col}_{slave}' for col in curr_engineered_slave_data.columns if col not in ['date_stamp', 'sku']}
        )  
        
        # Add 'quantity_sold_t-1_{slave}' column to slave data
        curr_engineered_slave_data = curr_engineered_slave_data.merge(lagged_series, on='date_stamp', how='inner')               
        
    ##########################################################################################################
    # RETIEVE MASTER FEATURES
    ##########################################################################################################
        
        # Build 'quantity_sold_{master}' and 'on_promotion_{master}' columns
        masterPromotionalPattern = sorted(list(masterPromotionalPattern))  # Sort SKUs in alphanumeric order
        master_pattern_features = (
            processed_data.loc[processed_data['sku'].isin(masterPromotionalPattern)]                 # Select data for the current masterPromotionalPattern
              .pivot(index='date_stamp', columns='sku', values=['opening_stock', 'on_promotion'])    # Pivot to wide format with 'opening_stock' and 'on_promotion' columns
              .swaplevel(axis=1)                                
              .sort_index(axis=1, level=0)                      
        )
        master_pattern_features.columns = [f'{val}_{sku}' for sku, val in master_pattern_features.columns]    # Rename columns to 'quantity_sold_{sku}' and 'on_promotion_{sku}'
        master_pattern_features = master_pattern_features[sorted(master_pattern_features.columns)]            # For consistency, align columns according to the SKUs order in masterPromotionalPattern
        master_pattern_features = master_pattern_features.reset_index().rename(columns={'index': 'date_stamp'})

    ##########################################################################################################
    # DEFINE THE TREATMENT VARIABLE
    ##########################################################################################################
    
        # Build the 'treatment' column: set to 1 when all masters in the pattern are on promotion, 0 when none are on promotion. Drop mixed cases
        on_promotion_cols = [c for c in master_pattern_features.columns if c.startswith('on_promotion_')]   
        treated = (master_pattern_features[on_promotion_cols] == 1).all(axis=1)                              
        untreated = (master_pattern_features[on_promotion_cols] == 0).all(axis=1)
        master_pattern_features['treatment'] = np.where(treated, 1, np.where(untreated, 0, np.nan))    

    ##########################################################################################################
    # RESTRICT DATA TO RELEVANT DATES
    ##########################################################################################################
        
        master_pattern_features = master_pattern_features.dropna(subset=['treatment']).drop(columns=on_promotion_cols).reset_index(drop=True)
        master_pattern_features['treatment'] = master_pattern_features['treatment'].astype(int)
        # Enrich slave data with 'quantity_sold_{master}' and 'treatment' columns
        curr_engineered_slave_data = curr_engineered_slave_data.merge(master_pattern_features, on='date_stamp')
                                  

        if (len(curr_engineered_slave_data['treatment'].unique()) < 2) or curr_engineered_slave_data['treatment'].value_counts().min() < MIN_SAMPLES_PER_GROUP:    # Check min samples constraint
            continue
        
    ##########################################################################################################
    # DEFINITION OF FEATURE SET X, TREATMENT VARIABLE T AND SALES Y
    ##########################################################################################################  
        # Identify the relevant features according to the EconML framework notation
        _X = curr_engineered_slave_data.drop(columns=['date_stamp', 'sku', 'treatment', f'quantity_sold_{slave}']).columns.tolist()
        _T = 'treatment'
        _Y = f'quantity_sold_{slave}'

        
##############################################################################################################
# PHASE 5: PROPENSITY-BASED SAMPLE TRIMMING
##############################################################################################################
        
        skip_to_next = False
        result, scores = positivity_check(T=curr_engineered_slave_data[_T], 
                                          X=curr_engineered_slave_data[_X], 
                                          alpha=0.05,
                                          random_state=SEED)        
        
        while not result:
            
            n_samples_before = len(curr_engineered_slave_data)
            # Trimming operation
            curr_engineered_slave_data = curr_engineered_slave_data.loc[
                np.where(((scores >= 0.05)) & ((scores <= 0.95)))].reset_index(drop=True)
            
            # Stop if trimming no longer reduces the sample while positivity is still violated,
            # indicating that common support cannot be recovered for the current dataset
            if len(curr_engineered_slave_data) == n_samples_before:
                skip_to_next = True
                break
            
            # Check min samples constraint
            if (len(curr_engineered_slave_data[_T].unique()) < 2
                or curr_engineered_slave_data[_T].value_counts().min() < MIN_SAMPLES_PER_GROUP):
                skip_to_next = True
                break
                
            result, scores = positivity_check(T=curr_engineered_slave_data[_T], 
                                              X=curr_engineered_slave_data[_X], 
                                              alpha=0.05,
                                              random_state=SEED)
        
        if skip_to_next:
            continue
            
##############################################################################################################
#                                       END Data Processing Methodology                                      #
##############################################################################################################
        
    

##############################################################################################################
#                                                 Placebo Test                                               #
##############################################################################################################

#    The placebo test is conducted under the following hypotheses.
#
#    Null hypothesis (H0):
#    The observed causal effect is drawn from the null distribution of effects
#    obtained by permuting the treatment T; no real causal effect exists.
#
#    Alternative hypothesis (H1):
#    The observed causal effect does not belong to the null distribution and
#    indicates a real causal effect (positive or negative).
#
#    p-value:
#    The p_value is the two-sided probability, under the null distribution,
#    of observing an effect at least as extreme as obs_estimate.
    
        print(f"Pair (master_pattern={masterPromotionalPattern}, slave='{slave}'): placebo test running")

        # Retrive the causal effects estimated in the current experiment
        obs_ate = outcomes.loc[
            (outcomes[('slave', '')] == slave) &
            (outcomes[('master_pattern', '')].apply(lambda x: x == masterPromotionalPattern))
        ].ATE.estimate.values[0]
        obs_att = outcomes.loc[
            (outcomes[('slave', '')] == slave) &
            (outcomes[('master_pattern', '')].apply(lambda x: x == masterPromotionalPattern))
        ].ATT.estimate.values[0]
        obs_atc = outcomes.loc[
            (outcomes[('slave', '')] == slave) &
            (outcomes[('master_pattern', '')].apply(lambda x: x == masterPromotionalPattern))
        ].ATC.estimate.values[0]
        
        n_simulations = 1000
        
        null_ate_list = []
        null_att_list = []
        null_atc_list = []
        
        original_curr_engineered_slave_data = curr_engineered_slave_data.copy()
        del curr_engineered_slave_data    # reset variable
        
        for simulation in range(0, n_simulations):   
            
            curr_engineered_slave_data = original_curr_engineered_slave_data.copy()
            # Permutation of the treatment column
            curr_engineered_slave_data[_T] = (
                curr_engineered_slave_data[_T]
                .sample(frac=1, random_state=SEED+simulation)
                .to_numpy()
            )
            
    ##########################################################################################################
    # CAUSAL ANALYSIS
    ##########################################################################################################  
            
            # Perform causal analysis using the Double Machine Learning (DML) procedure implemented by the EconML framework

            # Base heuristics for hyperparameters setting
            min_samples_leaf_gb = max(int(np.ceil(0.01 * len(curr_engineered_slave_data))), 2*len(_X))

            # Define model_y to fit the outcome (_Y) to the features (_X) in the DML procedure                 
            model_y = GradientBoostingRegressor(learning_rate=0.1,
                                                n_estimators=200,
                                                subsample=0.8,
                                                min_samples_split=int(2 * min_samples_leaf_gb),
                                                min_samples_leaf=min_samples_leaf_gb,
                                                max_depth=2,
                                                max_features=0.7, 
                                                random_state=SEED)  

            # Define model_t to fit the treatment (_T) to the features (_X) in the DML procedure
            model_t = GradientBoostingClassifier(learning_rate=0.1,
                                                 n_estimators=200,
                                                 subsample=0.8,
                                                 min_samples_split=int(2 * min_samples_leaf_gb),
                                                 min_samples_leaf=min_samples_leaf_gb,
                                                 max_depth=2,
                                                 max_features=0.7,
                                                 random_state=SEED)

            # Base heuristics for hyperparameters setting
            min_samples_leaf_cf = max(30, int(np.sqrt(len(curr_engineered_slave_data))))

            # Define dml_estimator to fit the response residuals to the treatment residuals
            dml_estimator = CausalForestDML(model_y=model_y, 
                                            model_t=model_t, 
                                            categories=[0,1],
                                            cv=5,    # cross-fitting parameter
                                            discrete_treatment=True,
                                            n_estimators=200,
                                            max_depth=None, 
                                            min_samples_split=int(2 * min_samples_leaf_cf),
                                            min_samples_leaf=min_samples_leaf_cf,
                                            max_features='sqrt',
                                            random_state=SEED)

            # Train dml_estimator
            dml_estimator.fit(Y=curr_engineered_slave_data[_Y], 
                              T=curr_engineered_slave_data[_T], 
                              X=curr_engineered_slave_data[_X])

            # Estimate the causal effects
            
            # Estimate ATE
            if pd.notna(obs_ate):
                _ate_inference = dml_estimator.ate_inference(X=curr_engineered_slave_data[_X])
                null_ate_list.append(_ate_inference.mean_point)
                
            # Estimate ATT
            if pd.notna(obs_att):
                _att_inference = dml_estimator.ate_inference(X=curr_engineered_slave_data.loc[curr_engineered_slave_data[_T]==1][_X])
                null_att_list.append(_att_inference.mean_point)
           
            # Estimate ATC 
            if pd.notna(obs_atc):
                _atc_inference = dml_estimator.ate_inference(X=curr_engineered_slave_data.loc[curr_engineered_slave_data[_T]==0][_X])
                null_atc_list.append(_atc_inference.mean_point)
        
        # Perform a bootstrap significance test using the original estimate and the set of simulations
        ate_placebo_estimate = np.nan
        ate_placebo_ci_lower, ate_placebo_ci_upper = np.nan, np.nan
        ate_placebo_p_value = np.nan        
        if pd.notna(obs_ate):
            ate_placebo_p_value = perform_bootstrap_test(obs_ate, null_ate_list)
            ate_placebo_estimate = np.mean(null_ate_list)
            ate_placebo_ci_lower = np.quantile(null_ate_list, 0.025)
            ate_placebo_ci_upper = np.quantile(null_ate_list, 0.975)

        att_placebo_estimate = np.nan
        att_placebo_ci_lower, att_placebo_ci_upper = np.nan, np.nan
        att_placebo_p_value = np.nan        
        if pd.notna(obs_att):
            att_placebo_p_value = perform_bootstrap_test(obs_att, null_att_list)
            att_placebo_estimate = np.mean(null_att_list)
            att_placebo_ci_lower = np.quantile(null_att_list, 0.025)
            att_placebo_ci_upper = np.quantile(null_att_list, 0.975)

            
        atc_placebo_estimate = np.nan
        atc_placebo_ci_lower, atc_placebo_ci_upper = np.nan, np.nan
        atc_placebo_p_value = np.nan        
        if pd.notna(obs_atc):
            atc_placebo_p_value = perform_bootstrap_test(obs_atc, null_atc_list)
            atc_placebo_estimate = np.mean(null_atc_list)
            atc_placebo_ci_lower = np.quantile(null_atc_list, 0.025)
            atc_placebo_ci_upper = np.quantile(null_atc_list, 0.975)
            
        # Collect outcomes
        effects_and_diagnostics = [obs_ate, ate_placebo_estimate, ate_placebo_ci_lower, ate_placebo_ci_upper, ate_placebo_p_value,
                                   obs_att, att_placebo_estimate, att_placebo_ci_lower, att_placebo_ci_upper, att_placebo_p_value,
                                   obs_atc, atc_placebo_estimate, atc_placebo_ci_lower, atc_placebo_ci_upper, atc_placebo_p_value]
        effects_and_diagnostics = [np.round(val, 3) if ii in [4, 9, 14] else np.round(val, 2) for ii, val in enumerate(effects_and_diagnostics)]    # Round p_values to 3 decimals, others to 2
          
        curr_placebo_outcomes = [[masterPromotionalPattern, slave] + effects_and_diagnostics]

        # Append the current placebo analysis to the global outcomes structure
        curr_placebo_outcomes = pd.DataFrame(curr_placebo_outcomes, columns=placebo_outcomes.columns)
        placebo_outcomes = pd.concat([placebo_outcomes, curr_placebo_outcomes], ignore_index=True)

        del curr_engineered_slave_data
        
times.append(time.perf_counter()) 
print('      Test duration (in minutes):', f"{np.round(sum([times[i+1] - times[i] for i in range(0, len(times)-1, 2)])/60,2):.2f}")
        
if placebo_outcomes.empty:
    raise SystemExit("The test terminates without producing any result") from None

09:45:33 - cmdstanpy - INFO - Chain [1] start processing
09:45:33 - cmdstanpy - INFO - Chain [1] done processing
09:45:33 - cmdstanpy - INFO - Chain [1] start processing
09:45:33 - cmdstanpy - INFO - Chain [1] done processing
09:45:33 - cmdstanpy - INFO - Chain [1] start processing
09:45:33 - cmdstanpy - INFO - Chain [1] done processing
09:45:34 - cmdstanpy - INFO - Chain [1] start processing
09:45:34 - cmdstanpy - INFO - Chain [1] done processing
09:45:34 - cmdstanpy - INFO - Chain [1] start processing
09:45:34 - cmdstanpy - INFO - Chain [1] done processing
09:45:34 - cmdstanpy - INFO - Chain [1] start processing
09:45:34 - cmdstanpy - INFO - Chain [1] done processing
09:45:34 - cmdstanpy - INFO - Chain [1] start processing
09:45:34 - cmdstanpy - INFO - Chain [1] done processing
09:45:34 - cmdstanpy - INFO - Chain [1] start processing
09:45:34 - cmdstanpy - INFO - Chain [1] done processing
09:45:34 - cmdstanpy - INFO - Chain [1] start processing
09:45:34 - cmdstanpy - INFO - Chain [1]

Pair (master_pattern=['10100', '10110'], slave='10106'): placebo test running
Pair (master_pattern=['10100', '10110'], slave='10126'): placebo test running
Pair (master_pattern=['10107'], slave='10119'): placebo test running
Pair (master_pattern=['10119', '10122', '10124'], slave='10129'): placebo test running
Pair (master_pattern=['10121', '10151', '10175', '10190', '10191'], slave='10133'): placebo test running
Pair (master_pattern=['10160'], slave='10170'): placebo test running
      Test duration (in minutes): 300.96


### Placebo test outcomes

In [12]:
confirmed_outcomes = None

if placebo_outcomes is not None and not placebo_outcomes.empty:
    # Align the ordering of placebo outcomes with the ordering used in the current experiment
    placebo_outcomes['_tmp_master_pattern'] = placebo_outcomes[('master_pattern', '')].apply(str)
    placebo_outcomes = placebo_outcomes.sort_values(by=['_tmp_master_pattern', ('slave', '')])   
    placebo_outcomes = placebo_outcomes.drop(columns='_tmp_master_pattern', level=0)

    # Assign rule_id
    placebo_outcomes[("rule_id", "")] = placebo_outcomes.apply(
        lambda row: generate_rule_id("_".join(map(str, row[("master_pattern", "")] + [row[("slave", "")]]))),
        axis=1
    )
    placebo_outcomes.set_index("rule_id", inplace=True)

    # Store placebo_outcomes
    placebo_outcomes.to_pickle("outcomes/sotadml_placebo_outcomes.pkl")

    # Select statistically significant causal effects based on placebo tests, 
    # retaining estimates with placebo p_value < 0.05
    significance_level = 0.05
    ate_idx = placebo_outcomes.index[placebo_outcomes.ATE.placebo_p_value < significance_level]
    att_idx = placebo_outcomes.index[placebo_outcomes.ATT.placebo_p_value < significance_level]
    atc_idx = placebo_outcomes.index[placebo_outcomes.ATC.placebo_p_value < significance_level]

    # Mask non-significant estimates (set to NaN) for each causal estimand
    for estimand, idx in {"ATE": ate_idx, "ATT": att_idx, "ATC": atc_idx}.items():
        outcomes.loc[~outcomes.index.isin(idx), (estimand, slice(None))] = np.nan

    # Drop rows where all ATE, ATT and ATC estimates are missing
    confirmed_outcomes = outcomes.dropna(how="all",
                                         subset=[(estimand, col) 
                                                 for estimand in ["ATE", "ATT", "ATC"] 
                                                 for col in outcomes[estimand].columns]).copy()

    # Store the confirmed causal estimates
    confirmed_outcomes.to_pickle("outcomes/sotadml_confirmed_outcomes.pkl")

    print('   Significant estimated effects:', f"{len(outcomes):>4d}")
    print('     Confirmed estimated effects:', f"{len(confirmed_outcomes):>4d}")

    print("\nPlacebo test outcomes")
    display(placebo_outcomes)
    print("\nConfirmed estimated effects")
    display(confirmed_outcomes)

   Significant estimated effects:    6
     Confirmed estimated effects:    6

Placebo test outcomes


master_pattern  slave      ATE                   \
                                                    estimate placebo_estimate   
rule_id                                                                         
Q5G                           [10100, 10110]  10106    -3.11            -0.02   
D30                           [10100, 10110]  10126    -2.03             0.02   
W6M                                  [10107]  10119    -2.33            -0.01   
BV9                    [10119, 10122, 10124]  10129    -1.04             0.01   
Z4R      [10121, 10151, 10175, 10190, 10191]  10133      NaN              NaN   
0SE                                  [10160]  10170    -0.86             0.00   

                                                               ATT  \
        placebo_ci_lower placebo_ci_upper placebo_p_value estimate   
rule_id                                                              
Q5G                -1.13             1.11             0.0    -2.96   
D30                -0.98             1.02             0.0    -2.13   
W6M                -1.17             1.14             0.0    -2.41   
BV9                -0.58             0.66             0.0    -1.06   
Z4R                  NaN              NaN             NaN     1.73   
0SE                -0.44             0.47             0.0      NaN   

                                                                            \
        placebo_estimate placebo_ci_lower placebo_ci_upper placebo_p_value   
rule_id                                                                      
Q5G                -0.02            -1.13             1.12             0.0   
D30                 0.02            -0.99             1.04             0.0   
W6M                -0.01            -1.18             1.13             0.0   
BV9                 0.01            -0.58             0.66             0.0   
Z4R                 0.01            -0.93             0.96             0.0   
0SE                  NaN              NaN              NaN             NaN   

             ATC                                                     \
        estimate placebo_estimate placebo_ci_lower placebo_ci_upper   
rule_id                                                               
Q5G        -3.15            -0.02            -1.14             1.11   
D30        -2.01             0.02            -0.97             1.02   
W6M        -2.30            -0.01            -1.17             1.14   
BV9        -1.04             0.01            -0.58             0.66   
Z4R          NaN              NaN              NaN              NaN   
0SE        -0.85             0.00            -0.44             0.47   

                         
        placebo_p_value  
rule_id                  
Q5G                 0.0  
D30                 0.0  
W6M                 0.0  
BV9                 0.0  
Z4R                 NaN  
0SE                 0.0


Confirmed estimated effects


master_pattern  slave      ATE           \
                                                    estimate ci_lower   
rule_id                                                                 
Q5G                           [10100, 10110]  10106    -3.11    -4.65   
D30                           [10100, 10110]  10126    -2.03    -3.27   
W6M                                  [10107]  10119    -2.33    -4.11   
BV9                    [10119, 10122, 10124]  10129    -1.04    -1.91   
Z4R      [10121, 10151, 10175, 10190, 10191]  10133      NaN      NaN   
0SE                                  [10160]  10170    -0.86    -1.71   

                                             ATT                            \
        ci_upper std_err z_stat p_value estimate ci_lower ci_upper std_err   
rule_id                                                                      
Q5G        -1.57    0.78  -3.97   0.000    -2.96    -4.57    -1.35    0.82   
D30        -0.79    0.63  -3.21   0.001    -2.13    -3.31    -0.95    0.60   
W6M        -0.55    0.91  -2.57   0.010    -2.41    -3.96    -0.87    0.79   
BV9        -0.17    0.44  -2.35   0.019    -1.06    -1.78    -0.33    0.37   
Z4R          NaN     NaN    NaN     NaN     1.73     0.15     3.32    0.81   
0SE        -0.01    0.43  -1.98   0.047      NaN      NaN      NaN     NaN   

                            ATC                                           \
        z_stat p_value estimate ci_lower ci_upper std_err z_stat p_value   
rule_id                                                                    
Q5G      -3.61   0.000    -3.15    -4.67    -1.64    0.77  -4.07   0.000   
D30      -3.53   0.000    -2.01    -3.26    -0.75    0.64  -3.13   0.002   
W6M      -3.06   0.002    -2.30    -4.17    -0.43    0.95  -2.41   0.016   
BV9      -2.85   0.004    -1.04    -1.94    -0.14    0.46  -2.27   0.023   
Z4R       2.15   0.032      NaN      NaN      NaN     NaN    NaN     NaN   
0SE        NaN     NaN    -0.85    -1.69    -0.01    0.43  -1.99   0.047   

        ate_robustness delta_sales_% delta_revenue_€ trimmed_sample  \
                                                                      
rule_id                                                               
Q5G              0.101        -26.33           -9.53           True   
D30              0.057        -20.30           -3.30           True   
W6M              0.048        -17.30           -3.30           True   
BV9              0.029        -28.70           -0.88           True   
Z4R                NaN           NaN             NaN          False   
0SE              0.024        -49.49           -1.41          False   

                                                                                                                                                                                                                                                                                                                         feature_importances  
                                                                                                                                                                                                                                                                                                                                              
rule_id                                                                                                                                                                                                                                                                                                                                       
Q5G                                                                                           {'trend_10106': 0.17, 'weekly_seasonality_10106': 0.09, 'yearly_seasonality_10106': 0.11, 'opening_stock_10106': 0.13, 'selling_price_10106': 0.09, 'quantity_sold_t-1_10106': 0.14, 'opening_stock_10100': 0.16, 'opening_stock_10110': 0.11}  
D30 

In [13]:
if confirmed_outcomes is not None and not confirmed_outcomes.empty:
    # Display results by replacing SKUs with their corresponding descriptions 
    results_with_desc = confirmed_outcomes.copy()
    sku_to_desc = dict(zip(descriptions['sku'].astype(str), descriptions['description']))      # Build mapping {sku: description}
    results_with_desc['master_pattern'] = results_with_desc['master_pattern'].apply(           # Replace SKUs in master_pattern with descriptions
        lambda x: [sku_to_desc[master] for master in x])
    results_with_desc['slave'] = results_with_desc['slave'].apply(lambda x: sku_to_desc[x])    # Replace SKUs in slave with descriptions
    display(results_with_desc)

master_pattern  \
                                                                                                                                                                          
rule_id                                                                                                                                                                   
Q5G                                                                                                                    [COCA COLA PET 1,5LT, COCA COLA ZERO PET LT1,5£]   
D30                                                                                                                    [COCA COLA PET 1,5LT, COCA COLA ZERO PET LT1,5£]   
W6M                                                                                                                                                    [COCA COLA LT.2]   
BV9                                                                                      [PEPSI REGULAR 1,5L.PET, PEPSI MAX S/CAFFEINA PET LT.1,5, PEPSI TWIST LT.1,5£]   
Z4R      [BEN COLA S.BENEDETTO PET LT1,5£, BIBITA ALLEGRA S.BENED.LT1,5, LIMONATA S.BENEDETTO PT LT.1,5, GASSOSA S.BENEDETTO PET LT.1,5, CEDRATA S.BENEDETTO PET LT1,5]   
0SE                                                                                                                                             [FANTA ORANGE PET CL66]   

                               slave      ATE                            \
                                     estimate ci_lower ci_upper std_err   
rule_id                                                                   
Q5G      COCA COLA PET LT1,35X2 BPK£    -3.11    -4.65    -1.57    0.78   
D30               COCA COLA LT1 PET£    -2.03    -3.27    -0.79    0.63   
W6M           PEPSI REGULAR 1,5L.PET    -2.33    -4.11    -0.55    0.91   
BV9          PEPSI COLA REG.PET CL50    -1.04    -1.91    -0.17    0.44   
Z4R        PEPSI-COLA BARATTOLO CL33      NaN      NaN      NaN     NaN   
0SE                 SPRITE LT1,5 PET    -0.86    -1.71    -0.01    0.43   

                            ATT                                           \
        z_stat p_value estimate ci_lower ci_upper std_err z_stat p_value   
rule_id                                                                    
Q5G      -3.97   0.000    -2.96    -4.57    -1.35    0.82  -3.61   0.000   
D30      -3.21   0.001    -2.13    -3.31    -0.95    0.60  -3.53   0.000   
W6M      -2.57   0.010    -2.41    -3.96    -0.87    0.79  -3.06   0.002   
BV9      -2.35   0.019    -1.06    -1.78    -0.33    0.37  -2.85   0.004   
Z4R        NaN     NaN     1.73     0.15     3.32    0.81   2.15   0.032   
0SE      -1.98   0.047      NaN      NaN      NaN     NaN    NaN     NaN   

             ATC                                          ate_robustness  \
        estimate ci_lower ci_upper std_err z_stat p_value                  
rule_id                                                                    
Q5G        -3.15    -4.67    -1.64    0.77  -4.07   0.000          0.101   
D30        -2.01    -3.26    -0.75    0.64  -3.13   0.002          0.057   
W6M        -2.30    -4.17    -0.43    0.95  -2.41   0.016          0.048   
BV9        -1.04    -1.94    -0.14    0.46  -2.27   0.023          0.029   
Z4R          NaN      NaN      NaN     NaN    NaN     NaN            NaN   
0SE        -0.85    -1.69    -0.01    0.43  -1.99   0.047          0.024   

        delta_sales_% delta_revenue_€ trimmed_sample  \
                                                       
rule_id                                                
Q5G            -26.33           -9.53           True   
D30            -20.30           -3.30           True   
W6M            -17.30           -3.30           True   
BV9            -28.70           -0.88           True   
Z4R               NaN             NaN          False   
0SE            -49.49           -1.41          False   

                                                              